# Парный бенч DeepFilterNet и Sidon в Colab

Один фиксированный test-набор M-AILABS + MUSAN. Обе модели получают **один и тот же 16 kHz mono сигнал**. Sidon получает исходный 16 kHz WAV; DeepFilterNet получает его детерминированную копию, повышенную до 48 kHz без добавления информации. Все выходы возвращаются к 16 kHz и оцениваются по одним и тем же эталонам. Если Sidon ещё не установлен, его результаты можно положить в указанную папку позже.

Запускай ячейки сверху вниз. Нужны доступ к архиву MUSAN и `ru_RU.tgz` в `MyDrive/audio-harness/raw/`, если речь ещё не распакована. Первый запуск DeepFilterNet может скачать веса.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
import subprocess
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'gdown', 'soundfile', 'scipy', 'pystoi'], check=True)


In [ ]:
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/audio-harness')
RAW = DATA_ROOT / 'raw'
MUSAN_DIR = RAW / 'musan'
NOISE_DIR = MUSAN_DIR / 'musan' / 'noise'
MAILABS_DIR = RAW / 'm_ailabs_ru'
EVAL_ROOT = DATA_ROOT / 'derived' / 'denoise_eval_test_16k'
MUSAN_FILE_ID = '1kngNXIxSjXTV2JtS9PAbE03yQcnx7iu_'
MAILABS_ARCHIVE = RAW / 'ru_RU.tgz'

SEED = 42
PAIR_COUNT = 100
TEST_BOOK = ('female', 'hajdurova', 'falshiviy_kupon')
EVAL_RATE = 16_000
DF_RATE = 48_000

RAW.mkdir(parents=True, exist_ok=True)
print('Drive:', DATA_ROOT)
print('Новый test-набор:', EVAL_ROOT)


## 1. MUSAN: скачать и достать только шумы

Архив определяется по содержимому (ZIP или TAR), а не по имени. WAV распаковываются в одну предсказуемую папку. Нераспознанный или неполный архив остаётся на Drive для диагностики; исходный архив удаляется лишь после успешной проверки результата.


In [ ]:
import shutil
import tarfile
import zipfile

ARCHIVE = MUSAN_DIR / 'musan_archive'
MUSAN_DIR.mkdir(parents=True, exist_ok=True)

def noise_relative(name):
    parts = Path(name.replace('\\', '/')).parts
    if 'noise' not in parts:
        return None
    tail = parts[parts.index('noise') + 1:]
    if not tail or any(part in ('', '.', '..') for part in tail):
        return None
    if not tail[-1].lower().endswith('.wav'):
        return None
    return Path(*tail)

def extract_noise(archive_path):
    count = 0
    if zipfile.is_zipfile(archive_path):
        with zipfile.ZipFile(archive_path) as archive:
            for info in archive.infolist():
                rel = noise_relative(info.filename)
                if rel is None or info.is_dir():
                    continue
                target = NOISE_DIR / rel
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as src, target.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
                count += 1
    elif tarfile.is_tarfile(archive_path):
        with tarfile.open(archive_path, 'r:*') as archive:
            for info in archive:
                rel = noise_relative(info.name)
                if rel is None or not info.isfile():
                    continue
                target = NOISE_DIR / rel
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.extractfile(info) as src, target.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
                count += 1
    else:
        raise RuntimeError(f'Архив не распознан: {archive_path}')
    return count

existing_noise = sorted(NOISE_DIR.rglob('*.wav'))
if len(existing_noise) >= 900:
    print(f'MUSAN уже распакован: {len(existing_noise)} WAV в {NOISE_DIR}')
else:
    legacy = MUSAN_DIR / 'musan.zip'
    if not ARCHIVE.exists() and legacy.exists():
        ARCHIVE = legacy
    if ARCHIVE.exists() and not (zipfile.is_zipfile(ARCHIVE) or tarfile.is_tarfile(ARCHIVE)):
        print('Неполный архив найден; скачиваю заново:', ARCHIVE)
        ARCHIVE.unlink()
    if not ARCHIVE.exists():
        import gdown
        url = f'https://drive.google.com/file/d/{MUSAN_FILE_ID}/view'
        print('Скачиваю MUSAN из Google Drive…')
        partial = MUSAN_DIR / 'musan_archive.part'
        partial.unlink(missing_ok=True)
        gdown.download(url, str(partial), quiet=False, fuzzy=True)
        assert partial.exists() and partial.stat().st_size > 0, 'Архив MUSAN не скачался.'
        partial.replace(ARCHIVE)
    assert ARCHIVE.exists() and ARCHIVE.stat().st_size > 0, 'Архив MUSAN не скачался.'
    extracted = extract_noise(ARCHIVE)
    existing_noise = sorted(NOISE_DIR.rglob('*.wav'))
    assert extracted > 0 and existing_noise, 'В архиве не нашлись WAV из noise/.'
    print(f'Извлечено {extracted} WAV; найдено {len(existing_noise)} WAV.')
    ARCHIVE.unlink()
    print('Архив удалён после проверки распаковки.')


## 2. Русская речь M-AILABS

Если `ru_RU/by_book` уже есть, распаковка пропускается. Иначе положи `ru_RU.tgz` в `MyDrive/audio-harness/raw/` и запусти ячейку снова.


In [ ]:
BOOKS_DIR = MAILABS_DIR / 'ru_RU' / 'by_book'
if not BOOKS_DIR.is_dir():
    assert MAILABS_ARCHIVE.is_file(), f'Загрузи архив сюда: {MAILABS_ARCHIVE}'
    MAILABS_DIR.mkdir(parents=True, exist_ok=True)
    print('Распаковываю M-AILABS…')
    subprocess.run(['tar', '-xzf', str(MAILABS_ARCHIVE), '-C', str(MAILABS_DIR)], check=True)
assert BOOKS_DIR.is_dir(), f'После распаковки не найдена папка {BOOKS_DIR}'
clean_dir = BOOKS_DIR / Path(*TEST_BOOK) / 'wavs'
clean_files = sorted(clean_dir.glob('*.wav'))
noise_files = sorted(NOISE_DIR.rglob('*.wav'))
assert len(clean_files) >= PAIR_COUNT, f'Нужно {PAIR_COUNT} WAV в {clean_dir}, найдено {len(clean_files)}'
assert noise_files, f'Не найдены шумы в {NOISE_DIR}'
print('Test-книга:', '/'.join(TEST_BOOK))
print('Чистых WAV:', len(clean_files), '| шумов MUSAN:', len(noise_files))


## 3. Зафиксировать общий test-набор

`falshiviy_kupon` — книга из проектного `test` split; старый `denoise_eval` с другой книгой не используется. Генерация создаёт 100 пар при 0, 5 и 10 dB. `noisy/*.wav` — канонический вход Sidon в 16 kHz. `df_input/*.wav` — тот же сигнал после единственного общего полифазного ресемплинга в 48 kHz для DeepFilterNet. Исходные файлы, seed и SNR фиксируются; неполный набор при повторном запуске пересобирается целиком.


In [ ]:
import csv
import random
from math import gcd

import numpy as np
import soundfile as sf
from scipy.signal import resample_poly

MANIFEST = EVAL_ROOT / 'manifest.csv'

def mono(audio):
    return audio.mean(axis=1) if audio.ndim == 2 else audio

def resample(audio, old_rate, new_rate):
    if old_rate == new_rate:
        return audio.astype(np.float32, copy=False)
    common = gcd(old_rate, new_rate)
    return resample_poly(audio, new_rate // common, old_rate // common).astype(np.float32)

def rms(audio):
    return float(np.sqrt(np.mean(audio.astype(np.float64) ** 2)))

def manifest_complete():
    if not MANIFEST.is_file():
        return False
    with MANIFEST.open(encoding='utf-8', newline='') as handle:
        rows = list(csv.DictReader(handle))
    if len(rows) != PAIR_COUNT or any(row.get('eval_rate') != str(EVAL_RATE) or row.get('test_book') != '/'.join(TEST_BOOK) for row in rows):
        return False
    return all(
        all((EVAL_ROOT / row[k]).is_file() for k in ('clean', 'noise', 'noisy', 'df_input'))
        for row in rows
    )

if manifest_complete():
    print('Готовый test-набор найден:', MANIFEST)
else:
    rng = random.Random(SEED)
    selected = rng.sample(clean_files, PAIR_COUNT)
    snrs = ([0, 5, 10] * ((PAIR_COUNT + 2) // 3))[:PAIR_COUNT]
    rng.shuffle(snrs)
    for folder in ('clean', 'noise', 'noisy', 'df_input'):
        (EVAL_ROOT / folder).mkdir(parents=True, exist_ok=True)

    rows = []
    for i, (clean_path, snr) in enumerate(zip(selected, snrs)):
        clean, clean_rate = sf.read(clean_path, dtype='float32')
        clean = resample(mono(clean), clean_rate, EVAL_RATE)
        assert len(clean) > 0 and rms(clean) > 0, f'Пустая запись: {clean_path}'
        noise_path = rng.choice(noise_files)
        noise, noise_rate = sf.read(noise_path, dtype='float32')
        noise = resample(mono(noise), noise_rate, EVAL_RATE)
        assert len(noise) > 0 and rms(noise) > 0, f'Пустой шум: {noise_path}'
        if len(noise) >= len(clean):
            start = rng.randrange(len(noise) - len(clean) + 1)
            noise = noise[start:start + len(clean)].copy()
        else:
            noise = np.tile(noise, (len(clean) + len(noise) - 1) // len(noise))[:len(clean)]
        noise *= rms(clean) / rms(noise) / (10 ** (snr / 20))
        noisy = clean + noise
        peak = max(1.0, float(np.max(np.abs(noisy))))
        clean, noise, noisy = clean / peak, noise / peak, noisy / peak

        item_id = f'{i:04d}_snr{snr}'
        paths = {kind: f'{kind}/{item_id}.wav' for kind in ('clean', 'noise', 'noisy', 'df_input')}
        for kind, audio, rate in (
            ('clean', clean, EVAL_RATE),
            ('noise', noise, EVAL_RATE),
            ('noisy', noisy, EVAL_RATE),
        ):
            sf.write(EVAL_ROOT / paths[kind], audio, rate, subtype='PCM_16')
        # Ресемплим именно сохранённый PCM16 noisy: оба входа происходят от одинаковых байтов.
        shared_noisy, _ = sf.read(EVAL_ROOT / paths['noisy'], dtype='float32')
        sf.write(EVAL_ROOT / paths['df_input'], resample(shared_noisy, EVAL_RATE, DF_RATE), DF_RATE, subtype='PCM_16')
        rows.append({
            'id': item_id, **paths, 'snr_db': snr, 'eval_rate': EVAL_RATE,
            'test_book': '/'.join(TEST_BOOK),
            'source_clean': str(clean_path.relative_to(MAILABS_DIR)),
            'source_noise': str(noise_path.relative_to(NOISE_DIR)),
        })

    temp_manifest = MANIFEST.with_suffix('.csv.part')
    with temp_manifest.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    temp_manifest.replace(MANIFEST)
    print('Создано пар:', len(rows), '| manifest:', MANIFEST)

with MANIFEST.open(encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))
assert len(rows) == PAIR_COUNT
for row in rows:
    for kind, expected_rate in (('clean', EVAL_RATE), ('noisy', EVAL_RATE), ('df_input', DF_RATE)):
        assert sf.info(EVAL_ROOT / row[kind]).samplerate == expected_rate, (row['id'], kind)
print('Проверены частоты всех входов и эталонов.')


## 4. DeepFilterNet в Python 3.11

Пакет ставится в отдельное окружение `/content/df311`. DeepFilterNet получает заранее подготовленные WAV 48 kHz. Код проверяет ожидаемую частоту модели и сохраняет один выход для каждого `id`.


In [ ]:
DF_PYTHON = Path('/content/df311/bin/python')
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'uv'], check=True)
if not DF_PYTHON.is_file():
    subprocess.run(['uv', 'venv', '--python', '3.11', '/content/df311'], check=True)
subprocess.run([
    'uv', 'pip', 'install', '--python', str(DF_PYTHON),
    'torch==2.3.1+cpu', 'torchaudio==2.3.1+cpu',
    '--index-url', 'https://download.pytorch.org/whl/cpu',
], check=True)
subprocess.run([
    'uv', 'pip', 'install', '--python', str(DF_PYTHON),
    'DeepFilterNet==0.5.6', 'soundfile',
], check=True)
subprocess.run([str(DF_PYTHON), '-c', 'import df, torch; print("DeepFilterNet OK; torch", torch.__version__)'], check=True)


In [ ]:
import textwrap
from IPython.display import Audio, display

runner = Path('/content/run_deepfilter_batch.py')
runner.write_text(textwrap.dedent('''\
    import csv
    import sys
    from pathlib import Path

    import soundfile as sf
    from df.enhance import enhance, init_df, load_audio, save_audio

    root = Path(sys.argv[1])
    output_dir = root / 'deepfilter'
    output_dir.mkdir(exist_ok=True)
    model, state, _ = init_df(log_file=None)
    assert state.sr() == 48000, f'Ожидали DeepFilterNet 48 kHz, получили {state.sr()}'
    with (root / 'manifest.csv').open(encoding='utf-8', newline='') as handle:
        rows = list(csv.DictReader(handle))
    for index, row in enumerate(rows, 1):
        source = root / row['df_input']
        assert sf.info(source).samplerate == state.sr()
        destination = output_dir / f"{row['id']}.wav"
        if not destination.is_file():
            audio, _ = load_audio(source, sr=state.sr())
            enhanced = enhance(model, state, audio)
            save_audio(destination, enhanced, state.sr())
        print(f"{index}/{len(rows)} {row['id']}")
'''), encoding='utf-8')

subprocess.run([str(DF_PYTHON), str(runner), str(EVAL_ROOT)], check=True)
print('DeepFilterNet outputs:', EVAL_ROOT / 'deepfilter')


## 5. Sidon на тех же входах

Sidon получает **только** `noisy/*.wav` из этого manifest: mono PCM16 16 kHz. Положи его выходы в `sidon/<id>.wav`. Допустима собственная частота выхода Sidon (обычно 48 kHz): оценочная ячейка сама приведёт её к 16 kHz. Никакой отдельной нормализации, VAD, подавления тишины или дополнительного denoise перед Sidon не делай.

Если у тебя есть рабочая команда для одного файла, впиши её в `SIDON_COMMAND`, оставив `{input}` и `{output}`. Команда запускается для каждого test-файла. Пустое значение просто пропускает запуск; можно положить готовые выходы вручную. Конкретный CLI зависит от установленной версии Sidon, поэтому здесь нет выдуманной команды.


In [ ]:
import shlex

SIDON_COMMAND = ''  # пример формы: 'python your_infer.py --input {input} --output {output}'
SIDON_OUT = EVAL_ROOT / 'sidon'
SIDON_OUT.mkdir(exist_ok=True)

if SIDON_COMMAND.strip():
    assert '{input}' in SIDON_COMMAND and '{output}' in SIDON_COMMAND
    template = shlex.split(SIDON_COMMAND)
    for index, row in enumerate(rows, 1):
        source = EVAL_ROOT / row['noisy']
        destination = SIDON_OUT / f"{row['id']}.wav"
        if destination.is_file():
            continue
        command = [part.replace('{input}', str(source)).replace('{output}', str(destination)) for part in template]
        subprocess.run(command, check=True)
        assert destination.is_file(), f'Sidon не сохранил {destination}'
        print(f"{index}/{len(rows)} {row['id']}")
else:
    print('Команда Sidon не задана. Ожидаемые выходы:', SIDON_OUT)


## 6. Парная оценка

Для каждого `id` используются один эталон 16 kHz, один noisy-вход и результаты обеих моделей. Оба выхода приводятся одним `resample_poly` к 16 kHz, сравниваются на одной длине и выравниваются по задержке в пределах ±100 ms одним правилом. Считаются SI-SDR и STOI относительно clean. Таблица появляется **только если есть все 100 результатов обеих моделей**; пропавшие файлы не скрываются отбором удобного подмножества. SI-SDR оценивает близость волны и может быть строг к генеративному вокодеру Sidon; STOI добавлен для оценки разборчивости речи. Скорость здесь не сравнивается: для неё нужны одинаковое железо и отдельный протокол.


In [ ]:
from scipy.signal import correlate, correlation_lags
from pystoi import stoi

DF_OUT = EVAL_ROOT / 'deepfilter'
missing = {
    model: [row['id'] for row in rows if not (directory / f"{row['id']}.wav").is_file()]
    for model, directory in (('DeepFilterNet', DF_OUT), ('Sidon', SIDON_OUT))
}
if any(missing.values()):
    for model, ids in missing.items():
        if ids:
            print(f'{model}: нет {len(ids)} файлов; первые id: {ids[:5]}')
    print('Полная парная оценка начнётся, когда будут готовы оба набора.')
else:
    def load_at_16k(path):
        audio, rate = sf.read(path, dtype='float32')
        audio = mono(audio)
        assert len(audio) > 0 and np.isfinite(audio).all(), f'Некорректный WAV: {path}'
        return resample(audio, rate, EVAL_RATE)

    def align(estimate, reference):
        # Одинаковое правило для обеих моделей; сдвиг не должен маскировать обрезанный файл.
        tolerance = round(EVAL_RATE * 0.1)
        assert abs(len(estimate) - len(reference)) <= tolerance, 'Длина выхода отличается от clean более чем на 100 ms'
        estimate = np.pad(estimate[:len(reference)], (0, max(0, len(reference) - len(estimate))))
        corr = correlate(estimate, reference, mode='full', method='fft')
        lags = correlation_lags(len(estimate), len(reference), mode='full')
        allowed = np.flatnonzero(np.abs(lags) <= tolerance)
        lag = int(lags[allowed[np.argmax(np.abs(corr[allowed]))]])
        if lag > 0:
            estimate = np.pad(estimate[lag:], (0, lag))
        elif lag < 0:
            estimate = np.pad(estimate[:lag], (-lag, 0))
        return estimate[:len(reference)], lag

    def si_sdr(estimate, reference):
        estimate = estimate.astype(np.float64) - float(np.mean(estimate))
        reference = reference.astype(np.float64) - float(np.mean(reference))
        target = np.dot(estimate, reference) / (np.dot(reference, reference) + 1e-12) * reference
        error = estimate - target
        return 10 * np.log10((np.dot(target, target) + 1e-12) / (np.dot(error, error) + 1e-12))

    scores = []
    for row in rows:
        clean = load_at_16k(EVAL_ROOT / row['clean'])
        assert len(clean) > EVAL_RATE // 2, f"Слишком короткий файл: {row['id']}"
        for model, path in (
            ('Noisy', EVAL_ROOT / row['noisy']),
            ('DeepFilterNet', DF_OUT / f"{row['id']}.wav"),
            ('Sidon', SIDON_OUT / f"{row['id']}.wav"),
        ):
            output, lag = align(load_at_16k(path), clean)
            scores.append({
                'id': row['id'], 'snr_db': row['snr_db'], 'model': model,
                'si_sdr_db': float(si_sdr(output, clean)),
                'stoi': float(stoi(clean, output, EVAL_RATE, extended=False)),
                'lag_ms': 1000 * lag / EVAL_RATE,
            })

    score_path = EVAL_ROOT / 'paired_scores.csv'
    with score_path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=scores[0].keys())
        writer.writeheader()
        writer.writerows(scores)
    for snr in ('all', '0', '5', '10'):
        print('SNR:', snr)
        for model in ('Noisy', 'DeepFilterNet', 'Sidon'):
            subset = [r for r in scores if r['model'] == model and (snr == 'all' or r['snr_db'] == snr)]
            print(f"  {model:14} n={len(subset):3} SI-SDR={np.mean([r['si_sdr_db'] for r in subset]):6.2f} dB  STOI={np.mean([r['stoi'] for r in subset]):.3f}")
    print('Парные результаты:', score_path)
